# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 clinical dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 clinicopathological dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
# Optionally, use numpy for EDA
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

- Each record set, field, and column is referenced by its `@id`, ensuring traceable and reproducible access.

In [ ]:
# List all available record sets and their @ids
record_sets = dataset.record_sets

print("Available record sets:")
for rs in record_sets:
    print(f"  RecordSet Name: {rs.name} | @id: {rs.id}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      - Field Name: {field.name} | @id: {field.id} | DataType: {field.data_type}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

- Use the `@id` of the record set and fields from the previous overview.

- This example loads each record set, prints column names, and previews the data.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nRecordSet @id: {record_set_id}")
    print("Columns:", df.columns.tolist())
    print(df.head(2))

# Select the first record set for further exploration
main_record_set_id = record_set_ids[0]
main_df = dataframes[main_record_set_id]

# Keep track of a numeric field @id for EDA
numeric_fields = [field.id for field in dataset.get_record_set(main_record_set_id).fields if field.data_type in ['Integer', 'Float', 'Number']]
print(f"Numeric fields (@id): {numeric_fields}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

- Outlier removal
- Normalization
- Grouping by key attributes

Ensure fields are referenced by their `@id`.

In [ ]:
# Choose a numeric field and a grouping field by their @id
if len(numeric_fields) > 0:
    numeric_field_id = numeric_fields[0]
else:
    numeric_field_id = None

# For grouping, pick the first categorical field
cat_fields = [field.id for field in dataset.get_record_set(main_record_set_id).fields if field.data_type == 'Text']
group_field_id = cat_fields[0] if len(cat_fields)>0 else None

if numeric_field_id is not None and numeric_field_id in main_df.columns:
    threshold = main_df[numeric_field_id].mean()  # Use mean as threshold example

    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped averages of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

- We'll show a histogram for the chosen numeric field, and a bar plot for the group averages.
- Make sure axes reference the correct column `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in main_df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=12, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Grouping barplot
    if group_field_id is not None and group_field_id in main_df.columns:
        grouped_means = main_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_means)
        plt.title(f"Mean {numeric_field_id} per {group_field_id}")
        plt.xticks(rotation=40)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the FAIR^2 dataset exploration.
- The notebook demonstrates how to load tabular clinical data using the Croissant schema and `mlcroissant`.
- Records, fields, columns, and visualizations were always referenced via their `@id`s, supporting reproducibility.
- Filtering, normalization, and grouping analyses help highlight important patterns and outliers in second primary colorectal cancer survivors.
- Further investigation into specific clinicopathological variables can be performed by referencing their `@id`s and extending EDA and visualization code.